# Tests for `plotter.py`

Run this notebook from the folder it lives in (next to `plotter.py`).
`plotter.py` locates the dataset on its own, so the data path does not depend
on the working directory.

Cells ending in `PASSED` check a fact automatically (they raise an
`AssertionError` if something is wrong). The plots at the end are visual checks
meant to be eyeballed.

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.getcwd())   # import plotter.py from this notebook's own folder
import numpy as np
import pandas as pd
import plotter

## 1. Metadata and helpers
These need no data file and run instantly.

In [ ]:
cols = plotter.imu_cols('hand')
assert len(cols) == 17
assert cols[0] == 'hand_temp'
assert 'hand_acc16_x' in cols and 'hand_gyro_z' in cols
print('imu_cols: PASSED')

In [ ]:
assert len(plotter.COLUMNS) == 54
assert plotter.COLUMNS[:3] == ['timestamp', 'activity_id', 'heart_rate']
print('COLUMNS: PASSED (54 columns)')

In [ ]:
assert plotter.NAME_TO_ID['walking'] == 4
assert plotter.ACTIVITIES[plotter.NAME_TO_ID['running']] == 'running'
print('NAME_TO_ID round-trip: PASSED')

## 2. Loading a subject

In [ ]:
df = plotter.load_subject('101')
assert df.shape[1] == 54
assert len(df) > 0
known = set(plotter.ACTIVITIES) | {0}   # 0 = transient
assert set(df['activity_id'].unique()).issubset(known)
print('load_subject: PASSED', df.shape)

## 3. `get_segment` behaviour
Correct window size, the `start` offset, and the three ways it should decline
These three ways are `unknown activity`, `activity not performed`, `window past the end`.

In [ ]:
seg = plotter.get_segment(df, 'lying', start=0, seconds=10)
assert seg is not None
assert len(seg) == 10 * plotter.SAMPLE_RATE   # 10 s at 100 Hz
print('window length: PASSED', len(seg), 'rows')

In [ ]:
seg0  = plotter.get_segment(df, 'lying', start=0,  seconds=5)
seg10 = plotter.get_segment(df, 'lying', start=10, seconds=5)
assert seg10['timestamp'].iloc[0] > seg0['timestamp'].iloc[0]
print('start offset: PASSED')

In [ ]:
assert plotter.get_segment(df, 'skydiving', 0, 10) is None
print('unknown activity -> None: PASSED')

In [ ]:
present = set(df['activity_id'].unique())
missing = [name for aid, name in plotter.ACTIVITIES.items() if aid not in present]
if missing:
    assert plotter.get_segment(df, missing[0], 0, 10) is None
    print('not-performed activity -> None: PASSED  (tested', repr(missing[0]) + ')')
else:
    print('subject did every activity; nothing to test here')

In [ ]:
assert plotter.get_segment(df, 'lying', start=999999, seconds=10) is None
print('window past end -> None: PASSED')

## 4. `compute_spectrum`
The FFT is its own function returning `(freqs, spectrum)`, so it can be tested
directly. A pure 3 Hz sine sampled at 100 Hz should peak at 3 Hz.

In [ ]:
fs = plotter.SAMPLE_RATE
t = np.arange(0, 10, 1 / fs)          # 10 seconds at 100 Hz
signal = np.sin(2 * np.pi * 3 * t)    # pure 3 Hz sine

freqs, spectrum = plotter.compute_spectrum(signal)
peak = freqs[np.argmax(spectrum)]
assert abs(peak - 3) < 0.2
print('compute_spectrum finds the 3 Hz peak: PASSED  (peak =', round(peak, 2), 'Hz)')

In [ ]:
# Punch a gap of NaNs into the same signal; compute_spectrum should clean it
# internally so the output has no NaN and the 3 Hz peak survives.
gappy = signal.copy()
gappy[100:110] = np.nan
freqs2, spec2 = plotter.compute_spectrum(gappy)
assert not np.isnan(spec2).any()
peak2 = freqs2[np.argmax(spec2)]
assert abs(peak2 - 3) < 0.2
print('compute_spectrum handles NaN gaps: PASSED  (peak =', round(peak2, 2), 'Hz)')

## 5. Visual checks
Eyeball these. Lying flat, running busy; the running spectrum should carry far
more energy in the 1-8 Hz band than lying.

In [ ]:
plotter.plot_activity('101', 'lying')

In [ ]:
plotter.plot_activity('101', 'running')

In [ ]:
plotter.plot_activity('101', 'walking', start=30, seconds=10)   # offset window

In [ ]:
plotter.plot_activity('101', 'running', magnitude=True)         # movement magnitude

### Single vs. multiple sensors
The four checks above show one sensor at a time; these show several sensors
side by side via plot_activity_multi.

In [ ]:
plotter.plot_activity_multi('101', 'running', sensors=('hand_acc16', 'ankle_acc16'))  # two sensors side by side

In [ ]:
plotter.plot_activity_multi('101', 'walking', sensors=('hand_acc16', 'chest_acc16', 'ankle_acc16'), magnitude=True)  # three sensors, magnitude

### Multiple participants
plot_subjects_multi puts one participant per panel, each with its own
window start.

In [ ]:
plotter.plot_subjects_multi(('101', '102'), 'walking', starts=(0, 30))  # per-subject start

In [ ]:
plotter.plot_subjects_multi(('101', '102', '103'), 'running', sensor='ankle_acc16', magnitude=True)

In [ ]:
plotter.plot_spectrum('101', 'running')

In [ ]:
plotter.plot_spectrum('101', 'lying')